# 02 · Fault injection — EMFI (electromagnetic)

**Electromagnetic fault injection** induces a fast field pulse with a coil
held over the target die — no electrical contact — to upset the chip at a
precise moment. On FaultyCat this is `cat.emfi`, and unlike crowbar it also
captures an **ADC trace** of the shot.

> ⚠️ **High voltage.** EMFI charges an HV capacitor and discharges it
> through the coil. The plastic shield is **mandatory**.

In [ ]:
import faultycat as fc
SIM = False                # True to dry-run without a board
RST_GP = None              # GP wired to target nRST; set once firmware has 'reset'
cat = fc.connect(simulator=SIM)
cat.emfi

## 1 · A single shot + ADC trace (external trigger)

`glitch()` = apply settings → `arm()` (HV charges) → `fire()`. With an
external trigger it waits in `WAITING` for the target's edge. Then capture
the ADC ring around the shot.

In [ ]:
cat.emfi.trigger  = 'ext_rising'
cat.emfi.delay_us = 100
cat.emfi.width_us = 10

try:
    cat.emfi.glitch(trigger_timeout_ms=5000)
except fc.EngineError as e:
    print('engine:', e)      # e.g. HV_NOT_CHARGED / TRIGGER_TIMEOUT
print(cat.emfi.status)

In [ ]:
fc.plot_trace(cat.emfi.capture(length=512));
cat.emfi.disarm()

## 2 · Hunting the glitch — **you** program success

As in ChipWhisperer, whether a shot *worked* is decided by **observing the
target and classifying**, not by any tool field. Loop over parameters,
glitch, read the target, and assign a group with your own condition. Edit
`classify()` for your target.

In [ ]:
def classify(resp: bytes) -> str:
    """YOUR success condition — the whole point lives here."""
    if b'root' in resp or b'OK' in resp:
        return 'success'
    if not resp:
        return 'reset'
    return 'normal'

In [ ]:
gc = fc.GlitchController(['delay', 'width'])
gc.set_range('delay', range(0, 200, 20)).set_range('width', range(1, 30, 3))

if cat.uart:
    cat.uart.open()

for p in gc.glitch_values():
    cat.emfi.trigger  = 'immediate'
    cat.emfi.delay_us = p['delay']
    cat.emfi.width_us = p['width']
    if RST_GP is not None:
        cat.target_reset(RST_GP)
    if cat.uart:
        cat.uart.reset_input()
    cat.emfi.glitch()
    resp = cat.uart.read_until(b'\n', timeout=0.1) if cat.uart else b''
    gc.add(classify(resp))

gc.counts()

In [ ]:
gc.plot(x='delay', y='width');

> **Faster, coarser alternative:** `cat.campaign('emfi')` sweeps on-device
> and streams firmware-side `fire_status` / `verify_status` — not your
> success. Use the observe-and-classify loop above for real hits.

In [ ]:
cat.close()